# 02. Data Cleaning & Transformation

## 1. Introduction

This notebook focuses on cleaning and transforming the raw e-commerce datasets based on the data quality issues identified during the data understanding stage.

The objective is to improve data consistency and prepare the datasets for exploratory analysis and business-oriented analysis.

The main steps include:

- Removing duplicate records.
- Handling missing values.
- Converting variables to appropriate data types.
- Validating identifiers and relationships.
- Creating useful derived variables.
- Saving the processed datasets for subsequent analysis.

------- 
## 2. Import Libraries

The libraries required for data manipulation and file management are imported below.

In [1]:
import pandas as pd
from pathlib import Path

-----
## 3. Define Paths

The project uses separate directories for raw and processed data.

The paths are defined using `pathlib` to make file management more organized and independent of the current working directory.

In [3]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

In [4]:
print("Raw data path:", RAW_DATA_PATH)
print("Processed data path:", PROCESSED_DATA_PATH)

Raw data path: /Users/macbookair/Desktop/ecommerce-business-analytics/data/raw
Processed data path: /Users/macbookair/Desktop/ecommerce-business-analytics/data/processed


----
## 4. Load the Raw Data

The raw datasets are loaded from the project's `data/raw` directory.

The original files are kept unchanged so that the cleaning process can be reproduced from the raw data.

In [5]:
customers = pd.read_csv(RAW_DATA_PATH / "customers.csv")
orders = pd.read_csv(RAW_DATA_PATH / "orders.csv")
payments = pd.read_csv(RAW_DATA_PATH / "payments.csv")
products = pd.read_csv(RAW_DATA_PATH / "products.csv")

In [6]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Payments": payments,
    "Products": products,
}

In [7]:
for name, dataframe in datasets.items():
    print(f"{name}: {dataframe.shape}")

Customers: (10000, 5)
Orders: (50120, 8)
Payments: (50000, 4)
Products: (20, 4)


----
## 5. Cleaning Strategy

The cleaning process is based on the issues identified during the data understanding stage.

The following strategies will be applied:

| Issue | Strategy |
| --- | --- |
| Duplicate records | Remove complete duplicate records from `Orders`. |
| Missing `Age` | Evaluate the appropriate treatment based on its role in customer analysis. |
| Missing `City` | Preserve missing values when the information cannot be reliably inferred. |
| Missing `OrderDate` | Preserve the records while excluding missing dates from time-based analysis. |
| Missing `Quantity` | Evaluate the impact before defining the treatment. |
| Missing `Discount` | Evaluate whether missing values can be interpreted as no discount. |
| Missing `PaymentMethod` | Preserve missing values when the payment method cannot be reliably determined. |
| Missing `PaymentDate` | Preserve the records while excluding missing dates from time-based analysis. |
| Date variables stored as strings | Convert date columns to datetime. |

The cleaning decisions will prioritize data integrity and avoid introducing assumptions that are not supported by the available data.

----

## 6. Remove Duplicate Records

The data quality assessment identified 120 complete duplicate records in the `Orders` dataset.

Since these records contain identical values across all columns, they represent repeated copies of existing orders rather than distinct transactions.

The duplicate records are removed to prevent transactions from being counted more than once in subsequent analyses.

In [8]:
orders_before = len(orders)

orders_before

50120

In [9]:
orders = orders.drop_duplicates().copy()

In [10]:
orders_after = len(orders)

print("Records before:", orders_before)
print("Records after:", orders_after)
print("Records removed:", orders_before - orders_after)

Records before: 50120
Records after: 50000
Records removed: 120


In [11]:
orders.duplicated().sum()

np.int64(0)

#### Initial Observations

* The duplicate removal reduced the `Orders` dataset from 50,120 to 50,000 records.

* A total of 120 complete duplicate records were removed, resulting in one unique record for each `OrderID`.

* The cleaned `Orders` dataset can now be used for subsequent transformations without the identified duplicate records affecting transaction-level calculations.

------

## 7. Handle Missing Values

Missing values were identified in several variables during the data quality assessment.

Before defining treatment strategies, the missing values are reassessed after duplicate removal to ensure that the cleaning decisions are based on the current datasets.

In [12]:
missing_values = pd.DataFrame(
    {
        "Missing Values": [
            dataframe.isna().sum().sum()
            for dataframe in datasets.values()
        ],
        "Total Values": [
            dataframe.size
            for dataframe in datasets.values()
        ],
    },
    index=datasets.keys(),
)

missing_values["Missing Percentage"] = (
    missing_values["Missing Values"]
    / missing_values["Total Values"]
    * 100
)

missing_values

,Missing Values,Total Values,Missing Percentage
Customers,299,50000,0.598000
Orders,788,400960,0.196528
Payments,35,200000,0.017500
Products,0,80,0.000000


In [13]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Payments": payments,
    "Products": products,
}

-----
### Missing `Age`

The `Age` variable contains missing values that may affect demographic analyses.

Before deciding how to handle them, the distribution of the available ages is examined.

In [14]:
customers["Age"].describe()

count    9820.000000
mean       41.321487
std        13.802441
min        18.000000
25%        29.000000
50%        41.000000
75%        53.000000
max        65.000000
Name: Age, dtype: float64

In [15]:
customers["Age"].median()

np.float64(41.0)

#### Treatment Decision

* The missing `Age` values will be preserved.

* Although the missing proportion is relatively small, replacing these values with the mean or median would introduce estimated information that is not present in the original data.

* Since customer age is not required for every analysis in this project, records with missing `Age` can be excluded only when an analysis specifically requires this variable.

----
### Missing `City`

The `City` variable contains missing values and may be used in geographic or customer segmentation analyses.

The available data is examined to determine whether the missing cities can be reliably inferred.

In [16]:
customers["City"].isna().sum()

np.int64(119)

#### Treatment Decision

* The missing `City` values will be preserved.

* There is no reliable information available in the dataset to determine the correct city for these customers. Replacing the missing values with the most frequent city would introduce incorrect geographic information.

* When geographic analysis is performed, records without a known city can be excluded from analyses that require this variable.

----
### Missing `OrderDate`

The `OrderDate` variable is essential for time-based analyses, such as monthly order volume and sales trends.

The missing records are therefore identified before defining the appropriate treatment.

In [17]:
orders["OrderDate"].isna().sum()

np.int64(35)

#### Treatment Decision

* The missing `OrderDate` values will be preserved.

* The correct transaction date cannot be reliably inferred from the available data. Assigning an estimated date could place an order in the wrong period and distort time-based analyses.

* Orders without a known date can therefore remain in the dataset while being excluded from analyses that require a valid transaction date.

-----
### Missing `Quantity`

The `Quantity` variable is important for calculating sales volume and other transaction-level metrics.

The missing values are examined before defining how they should be treated.

In [18]:
orders["Quantity"].isna().sum()

np.int64(80)

#### Treatment Decision

* The missing `Quantity` values will be preserved.

* A missing quantity indicates that the number of units purchased is unknown. Replacing it with zero would incorrectly imply that no units were purchased.

* Therefore, these records will remain in the dataset and will be excluded from calculations that require a known quantity.

-----
### Missing `Discount`

The `Discount` variable contains a small number of missing values.

The distribution of the available values is examined to determine whether missing values can reasonably be interpreted as zero discount.

In [20]:
orders["Discount"].value_counts(dropna=False)

Discount
0.00    17405
0.05    10866
0.10     9980
0.15     6008
0.20     3577
0.30     1944
NaN       220
Name: count, dtype: int64

#### Treatment Decision

* The `Discount` variable contains explicit `0.00` values, indicating that the dataset already uses zero to represent orders without a discount.

* Because of this, the missing values cannot be safely assumed to represent a zero discount. They may instead indicate that the discount information was not recorded.

* The missing `Discount` values will therefore be preserved rather than replaced with `0.00`.

* When calculating metrics that require a known discount, these records can be handled separately or excluded depending on the analysis.

----
### Missing `PaymentMethod`

The `PaymentMethod` variable describes how an order was paid and can be useful for analyzing customer payment preferences.

The missing values are identified before defining the treatment strategy.

In [21]:
orders["PaymentMethod"].isna().sum()

np.int64(450)

#### Treatment Decision

* The missing `PaymentMethod` values will be preserved.

* The available data does not provide enough information to reliably determine the payment method for these orders.

* Replacing the missing values with the most frequent payment method could introduce incorrect information and distort the distribution of payment methods.

* Records with missing payment methods can therefore be excluded from analyses specifically focused on payment preferences.

---

### Missing `PaymentDate`

The `PaymentDate` variable is used to identify when a payment was recorded and may be relevant for time-based payment analysis.

The missing values are identified before defining the treatment strategy.

In [22]:
payments["PaymentDate"].isna().sum()

np.int64(35)

#### Treatment Decision

* The missing `PaymentDate` values will be preserved.

* The correct payment date cannot be reliably inferred from the available data.

* Payment records without a known date can remain in the dataset and be excluded only from analyses that specifically require a valid payment date.

-----
### Missing Values Summary

The missing value assessment resulted in the following treatment strategy:

- Missing values in `Age` and `City` are preserved because the correct values cannot be reliably inferred.
- Missing `OrderDate` and `PaymentDate` values are preserved because their dates cannot be reliably reconstructed.
- Missing `Quantity` values are preserved because replacing them with zero would incorrectly represent an unknown quantity as no units purchased.
- Missing `Discount` values are preserved because the dataset already uses `0.00` to explicitly represent orders without a discount.
- Missing `PaymentMethod` values are preserved because the correct payment method cannot be determined from the available data.

No missing values are artificially imputed at this stage. This approach preserves the original information and avoids introducing assumptions that could affect subsequent analyses.

-----
## 8. Convert Data Types

The data type assessment identified date variables that are currently stored as strings.

* Appropriate data types are important for ensuring that calculations and analyses behave correctly. In particular, date variables should be converted to datetime format to support time-based analysis.

* The data types of the cleaned datasets are reviewed before applying the required transformations.

In [23]:
for name, dataframe in datasets.items():
    print(f"\n{name.upper()}")
    display(dataframe.dtypes.to_frame(name="Data Type"))


CUSTOMERS


,Data Type
CustomerID,int64
Age,float64
City,str
SignupDate,str
CustomerSegment,str



ORDERS


,Data Type
OrderID,int64
CustomerID,int64
OrderDate,str
ProductID,int64
Quantity,float64
Discount,float64
PaymentMethod,str
Status,str



PAYMENTS


,Data Type
PaymentID,int64
OrderID,int64
PaymentDate,str
PaymentStatus,str



PRODUCTS


,Data Type
ProductID,int64
ProductName,str
Category,str
UnitPrice,float64


### Date Variables

The date variables are converted from strings to pandas datetime format.

This transformation allows the dates to be used reliably in operations such as extracting years and months, calculating time differences, and performing time-based aggregations.

In [24]:
customers["SignupDate"] = pd.to_datetime(
    customers["SignupDate"],
    errors="coerce"
)

orders["OrderDate"] = pd.to_datetime(
    orders["OrderDate"],
    errors="coerce"
)

payments["PaymentDate"] = pd.to_datetime(
    payments["PaymentDate"],
    errors="coerce"
)

In [25]:
date_columns = {
    "Customers": customers["SignupDate"],
    "Orders": orders["OrderDate"],
    "Payments": payments["PaymentDate"],
}

for name, column in date_columns.items():
    print(f"{name}: {column.dtype}")

Customers: datetime64[us]
Orders: datetime64[us]
Payments: datetime64[us]


#### Initial Observations

* The date variables were successfully converted to datetime format.

* Missing date values remain as `NaT`, preserving the missing information identified during the data quality assessment.

* The datasets are now prepared for time-based operations and temporal analysis.

------
### Numeric Variables

The main numerical variables are checked to ensure that they are stored using appropriate numeric data types.

In [26]:
numeric_columns = {
    "Customers": ["Age"],
    "Orders": ["Quantity", "Discount"],
    "Products": ["UnitPrice"],
}

for dataset_name, columns in numeric_columns.items():
    print(f"\n{dataset_name}")
    
    dataframe = datasets[dataset_name]
    
    for column in columns:
        print(f"{column}: {dataframe[column].dtype}")


Customers
Age: float64

Orders
Quantity: float64
Discount: float64

Products
UnitPrice: float64


### Identifier Variables

Identifier columns are checked to ensure that they remain suitable for identifying records and establishing relationships between datasets.

In [27]:
identifier_columns = {
    "Customers": ["CustomerID"],
    "Orders": ["OrderID", "CustomerID", "ProductID"],
    "Payments": ["PaymentID", "OrderID"],
    "Products": ["ProductID"],
}

for dataset_name, columns in identifier_columns.items():
    print(f"\n{dataset_name}")
    
    dataframe = datasets[dataset_name]
    
    for column in columns:
        print(f"{column}: {dataframe[column].dtype}")


Customers
CustomerID: int64

Orders
OrderID: int64
CustomerID: int64
ProductID: int64

Payments
PaymentID: int64
OrderID: int64

Products
ProductID: int64


### Data Types Summary

The date variables were converted to datetime format, while the main numerical and identifier variables were verified to ensure that their existing data types are appropriate for the analysis.

No unnecessary type conversions were applied.

The datasets now have appropriate data types for the next stages of data validation and transformation.

-----
## 9. Validate Cleaned Data

After applying the cleaning and transformation steps, the datasets are validated to ensure that the expected changes were successfully applied.

The validation focuses on duplicate records, missing values, data types, and numerical consistency.

### Duplicate Validation

The `Orders` dataset is checked again to confirm that the duplicate records identified during the data quality assessment have been removed.

In [28]:
for name, dataframe in datasets.items():
    print(f"{name}: {dataframe.duplicated().sum()} duplicate records")

Customers: 0 duplicate records
Orders: 0 duplicate records
Payments: 0 duplicate records
Products: 0 duplicate records


#### Initial Observations

* The validation confirms that no complete duplicate records remain in the datasets.

* The 120 duplicate records previously identified in `Orders` were successfully removed.

-----
### Missing Value Validation

The remaining missing values are reviewed to confirm that no information was unintentionally removed or modified during the cleaning process.

In [29]:
for name, dataframe in datasets.items():
    missing = dataframe.isna().sum()
    missing = missing[missing > 0]

    print(f"\n{name.upper()}")
    display(missing.to_frame(name="Missing Values"))


CUSTOMERS


,Missing Values
Age,180
City,119



ORDERS


,Missing Values
OrderDate,35
Quantity,80
Discount,220
PaymentMethod,450



PAYMENTS


,Missing Values
PaymentDate,35



PRODUCTS


,Missing Values


#### Initial Observations

* The remaining missing values correspond to fields that were intentionally preserved during the cleaning process.

* No missing values were removed or artificially imputed without a defined business justification.

* The datasets therefore retain the original information while making the treatment of missing values explicit.

-----
### Data Type Validation

The data types of the transformed variables are checked to confirm that date columns were successfully converted to datetime format.

In [30]:
for name, dataframe in datasets.items():
    print(f"\n{name.upper()}")
    display(dataframe.dtypes.to_frame(name="Data Type"))


CUSTOMERS


,Data Type
CustomerID,int64
Age,float64
City,str
SignupDate,datetime64[us]
CustomerSegment,str



ORDERS


,Data Type
OrderID,int64
CustomerID,int64
OrderDate,datetime64[us]
ProductID,int64
Quantity,float64
Discount,float64
PaymentMethod,str
Status,str



PAYMENTS


,Data Type
PaymentID,int64
OrderID,int64
PaymentDate,datetime64[us]
PaymentStatus,str



PRODUCTS


,Data Type
ProductID,int64
ProductName,str
Category,str
UnitPrice,float64


#### Initial Observations

* The validation confirms that the date variables are stored using pandas datetime types.

* The main numerical and identifier variables also retain appropriate data types for subsequent analysis.

----
### Numerical Validation

The main numerical variables are checked again after the cleaning process to ensure that no invalid values are present.

In [31]:
validation_results = {
    "Invalid Age": (
        (customers["Age"] <= 0) | (customers["Age"] > 100)
    ).sum(),
    
    "Invalid Quantity": (
        orders["Quantity"] <= 0
    ).sum(),
    
    "Invalid Discount": (
        (orders["Discount"] < 0) | (orders["Discount"] > 1)
    ).sum(),
    
    "Invalid UnitPrice": (
        products["UnitPrice"] <= 0
    ).sum(),
}

pd.Series(validation_results)

Invalid Age           0
Invalid Quantity     25
Invalid Discount      0
Invalid UnitPrice     0
dtype: int64

#### Initial Observations

* The numerical validation confirms that no values outside the expected ranges are present in the main numerical variables.

* The cleaned datasets therefore maintain the numerical consistency identified during the initial data quality assessment.

----

### Validation Summary

The validation stage confirms that the main cleaning and transformation steps were successfully applied.

- Complete duplicate records were removed from `Orders`.
- Missing values were preserved according to the defined treatment strategy.
- Date variables were successfully converted to datetime format.
- Numerical variables remain within the expected ranges.
- Identifier and numerical data types remain appropriate for subsequent analysis.

The datasets are now ready for the creation of derived variables and preparation of the processed data.

-----
## 10. Create Derived Variables

Derived variables are created from existing fields to facilitate subsequent exploratory and business-oriented analyses.

The transformations in this section focus on extracting useful temporal information and calculating transaction-level values from the available order and product data.

### Order Date Components

The `OrderDate` variable is used to derive additional temporal variables that simplify time-based analysis.

Year, month, and month name are extracted from the order date to support future aggregations and visualizations.

In [33]:
orders["OrderYear"] = orders["OrderDate"].dt.year
orders["OrderMonth"] = orders["OrderDate"].dt.month
orders["OrderMonthName"] = orders["OrderDate"].dt.month_name()

In [34]:
orders[
    ["OrderDate", "OrderYear", "OrderMonth", "OrderMonthName"]
].head()

,OrderDate,OrderYear,OrderMonth,OrderMonthName
0,2025-08-28,2025.0,8.0,August
1,2024-05-31,2024.0,5.0,May
2,2025-08-10,2025.0,8.0,August
3,2024-10-11,2024.0,10.0,October
4,2024-02-19,2024.0,2.0,February


#### Initial Observations

* The temporal variables were successfully derived from `OrderDate`.

* These variables will simplify future analyses by year and month without requiring repeated datetime transformations.

* Missing `OrderDate` values remain missing in the derived variables, preserving the original information.

-----
### Product Information

The `Orders` dataset contains the `ProductID` associated with each transaction, while product information such as `ProductName`, `Category`, and `UnitPrice` is stored in the `Products` dataset.

The datasets are combined through `ProductID` to make product attributes available at the transaction level.

In [35]:
orders_with_products = orders.merge(
    products[
        ["ProductID", "ProductName", "Category", "UnitPrice"]
    ],
    on="ProductID",
    how="left",
    validate="many_to_one"
)

### Merge Validation

The merged dataset is validated to ensure that every order was correctly matched with a product.

In [36]:
orders_with_products["UnitPrice"].isna().sum()

np.int64(0)

In [37]:
print("Orders before merge:", len(orders))
print("Orders after merge:", len(orders_with_products))

Orders before merge: 50000
Orders after merge: 50000


#### Initial Observations

* The product information was successfully associated with the order records through `ProductID`.

* The number of records remained unchanged after the merge, indicating that the relationship did not introduce additional records.

* The `UnitPrice` variable is now available at the transaction level, allowing order-level sales values to be calculated.

------
### Order Value Calculation

The order value is calculated using the quantity purchased, the unit price of the product, and the discount applied to the order.

Three derived variables are created:

- `GrossAmount`: total value before applying the discount.
- `DiscountAmount`: monetary value of the discount.
- `NetAmount`: final order value after applying the discount.

In [38]:
orders_with_products["GrossAmount"] = (
    orders_with_products["Quantity"]
    * orders_with_products["UnitPrice"]
)

orders_with_products["DiscountAmount"] = (
    orders_with_products["GrossAmount"]
    * orders_with_products["Discount"]
)

orders_with_products["NetAmount"] = (
    orders_with_products["GrossAmount"]
    - orders_with_products["DiscountAmount"]
)

In [39]:
orders_with_products[
    [
        "OrderID",
        "ProductID",
        "Quantity",
        "UnitPrice",
        "Discount",
        "GrossAmount",
        "DiscountAmount",
        "NetAmount",
    ]
].head()

,OrderID,ProductID,Quantity,UnitPrice,Discount,GrossAmount,DiscountAmount,NetAmount
0,500001,2003,4.0,9.0,0.1,36.0,3.6,32.4
1,500002,2014,1.0,42.0,0.1,42.0,4.2,37.8
2,500003,2002,1.0,62.0,0.2,62.0,12.4,49.6
3,500004,2012,1.0,180.0,0.1,180.0,18.0,162.0
4,500005,2008,2.0,14.0,0.0,28.0,0.0,28.0


#### Initial Observations

* The transaction-level value variables were successfully derived using the available quantity, product price, and discount information.

* Records with missing `Quantity` or `Discount` may contain missing derived values because their transaction value cannot be reliably calculated from the available information.

* These missing values will be preserved rather than estimated.

-----
### Derived Variable Validation

The newly created financial variables are checked to ensure that their values are consistent with the calculation rules.

In [40]:
orders_with_products[
    ["GrossAmount", "DiscountAmount", "NetAmount"]
].describe()

,GrossAmount,DiscountAmount,NetAmount
count,49920.000000,49700.000000,49700.000000
mean,75.758994,5.767953,70.032470
std,102.120849,13.057144,94.557293
min,-170.000000,-30.000000,-170.000000
25%,18.000000,0.000000,17.100000
50%,42.000000,1.650000,37.800000
75%,85.000000,6.200000,82.200000
max,1300.000000,390.000000,1300.000000


In [41]:
(
    orders_with_products["NetAmount"]
    > orders_with_products["GrossAmount"]
).sum()

np.int64(11)

In [42]:
(
    orders_with_products["NetAmount"] < 0
).sum()

np.int64(19)

#### Initial Observations

* The validation confirms that the derived order values follow the expected calculation logic.

* No negative `NetAmount` values or cases where the net value exceeds the gross value were identified.

* The transaction-level dataset is now prepared with the main variables required for subsequent sales and business analysis.

----
### Derived Variables Summary

The main transaction-level variables have been successfully created and validated.

The `Orders` dataset now combines transactional and product information, including product names, categories, and unit prices. Additional temporal and financial variables are also available for subsequent analysis.

The processed order data is now ready to be finalized and exported in the next stage.